In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

from sympy import im
import torch 
import torch.nn as nn
import torch.optim as optim

SEED = 42


In [ ]:
w = torch.tensor(2.0, requires_grad=True)

y = 4 * w + 2

y.backward()

w.grad


tensor(4.)

In [10]:
x = torch.tensor([-3.0, -1.0, 0, 1.0, 3.0], requires_grad=True)

In [14]:
activations = {
    "ReLU": nn.ReLu(),
    "LeakyRelu": nn.LeakyReLU(negative_slope=0.1),
    "Tanh": nn.Tanh(),
    "Swiss": nn.SiLU()
}

for name, fn in activations.items():
    y = fn(x).numpy().round(3)
    rows.append(pd.Series(y, index=y.numpy(), name = name))

df - pd.DataFrame(rows)
df 

AttributeError: module 'torch.nn' has no attribute 'ReLu'

In [15]:
from torch import dropout


dropout = nn.Dropout(0.5)

data = torch.ones(16).reshape(4, 4)
data

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [ ]:
dropout.train()
dropout.eval()

data_y = dropout(data)
data_y

tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])

In [20]:
X, y = make_regression(
    n_features=20,        # 20 ознак
    n_informative=8,      # 8 з них інформативні
    noise=20,             # шум 20
    random_state=SEED
)


y = y.reshape(-1, 1)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=SEED
)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)


In [35]:
from scipy import optimize


model = nn.Sequential(
    nn.Linear(20, 32),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Dropout(0.2),

    nn.Linear(16, 1),

)

loss_fn = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=0.01)

In [39]:
epochs = 5000


history = []
for epochs in range(1, epochs +1):
    model.train()

    optimizer.zero_grad()
    
    y_pred_tensor = model(X_train_tensor)

    loss = loss_fn(y_pred_tensor, y_train_tensor)

    loss.backward()

    optimizer.step()
    loss_train = loss.item

    model.eval()

    with torch.no_grad():
        y_test_pred_tensor = model(X_test_tensor)
        loss_test = loss_fn(y_test_tensor, y_test_pred_tensor)

        history.append({
            "epoch": epochs,
            "train_loss": loss_train,
            "test_loss": loss_test.item()
        })




In [40]:
df =pd.DataFrame(history)
df.tail()

,epoch,train_loss,test_loss
4995,4996,<built-in method item of Tensor object at 0x00...,2334.692383
4996,4997,<built-in method item of Tensor object at 0x00...,2285.074219
4997,4998,<built-in method item of Tensor object at 0x00...,2264.158691
4998,4999,<built-in method item of Tensor object at 0x00...,2237.715332
4999,5000,<built-in method item of Tensor object at 0x00...,2190.430664
